# voice

> Speech to text on this machine, for think-aloud answers.

This module turns a spoken answer into text. With the local engines, the audio does not go off the machine.
The web app records while you talk. It cuts the audio into segments at your pauses and sends each segment here.
This module transcribes the segments one at a time, in order. The text goes into the think-aloud box while you continue to talk.

In [ ]:
#| default_exp voice

In [ ]:
#| hide
from fastcore.test import *

In [ ]:
#| hide
import os, tempfile
os.environ.setdefault('LITESEARCH_CACHE', tempfile.mkdtemp(prefix='ls-'))   # and litesearch does not wait on the shared lock of another run
os.environ['VISHALAKSHI_VAULT'] = os.environ['ABHYASA_VAULT'] = os.path.join(tempfile.mkdtemp(), 'vault.db')   # a test that opens the default vault gets a temporary one
os.environ['ABHYASA_CONFIG'] = tempfile.mkdtemp()   # and the settings and keys go to a temporary folder
os.environ['RAMABANA_CONFIG'] = os.path.join(tempfile.mkdtemp(), 'config.toml')   # and ramabana reads no real config
for k in ('ABHYASA_STT', 'OPENAI_API_KEY'): os.environ.pop(k, None)

These are the imports of the module.

In [ ]:
#| export
import os, re, gc, json, time, uuid, inspect, tempfile, threading, wave
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from fastcore.utils import *
from abhyasa.prompts import register, prompt

## Engines

There are three engines. `gemma` and `mlx` run on this machine. `openai` sends the audio to OpenAI.

- `gemma` runs a Gemma 4 model on LiteRT through the model layer. E4B is the default. E2B is faster.
- `mlx` runs any speech to text model that mlx-audio can load: Whisper, Parakeet, Qwen3-ASR, Voxtral, Canary, Moonshine and more.
  Install it with `pip install abhyasa[whisper]`.
- `openai` uses the OpenAI transcription API. It is off until you set a key. It sends the audio off the machine.

Each engine gets a hint: the name of the concept and its terms. Gemma reads it in its prompt. Whisper reads it as its initial prompt,
and Qwen3-ASR as hot words. OpenAI reads it as its prompt.

The table gives the numbers on this machine (an M-series Mac), with each model warm.
"Clip A" is the English test clip of the model layer (`rishi/nbs/speech.wav`).
"Clip B" is a clip in a cloned voice with Nyāya terms in IAST and technical terms.
"Terms" is the fraction of the 9 key terms in clip B that the engine wrote correctly.
"5 s" is the time for one 5-second segment.

| model | load | 5 s | clip A | clip B, no hint / hint | terms, no hint / hint | Sanskrit CER | memory |
|---|---|---|---|---|---|---|---|
| `litert-community/gemma-4-E4B-it-litert-lm` | 5.0 s | 0.69 s | 9% | 0% / 0% | 9 / 9 | 12% | 1.5 GB |
| `litert-community/gemma-4-E2B-it-litert-lm` | 2.6 s | 0.38 s | 14% | 6% / 2% | 7 / 8 | 13% | 1.3 GB |
| `mlx-community/whisper-large-v3-turbo-asr-fp16` | 4.8 s | 0.68 s | 6% | 4% / 0% | 8 / 9 | 10% | 2.3 GB |
| `mlx-community/parakeet-tdt-0.6b-v3` | 4.9 s | 0.15 s | 6% | 4% / 4% | 7 / 7 | Latin script | 3.2 GB |
| `mlx-community/Qwen3-ASR-0.6B-8bit` | 3.8 s | 0.14 s | 6% | 13% / 0% | 5 / 9 | 16% | 2.0 GB |
| `mlx-community/Qwen3-ASR-1.7B-8bit` | 3.6 s | 0.25 s | 6% | 6% / 0% | 7 / 9 | 14% | 3.6 GB |
| `mlx-community/Voxtral-Mini-4B-Realtime-2602-4bit` | 3.7 s | 2.21 s | 6% | 6% / 6% | 7 / 7 | 22% | 5.5 GB |
| `UsefulSensors/moonshine-base` | 3.9 s | 0.07 s | 100% | 8% / 8% | 6 / 6 | Latin script | 0.8 GB |
| `Mediform/canary-1b-v2-mlx-q8` | 4.2 s | 0.08 s | 6% | 4% / 4% | 8 / 8 | Latin script | 1.4 GB |
| Su-śrotā (NeMo, CPU, in vachaka) | 46 s | 0.3 s for 9 s | Devanagari only | Devanagari only | — | 0% | 1.7 GB |

- "load" is a cold start of a cached model in a new process, with the Python imports. The first download takes longer.
- "memory" is the MLX peak for the `mlx` models, and the growth of the process for Gemma on LiteRT.
- "Sanskrit CER" is the character error rate on a 9-second Nyāya sūtra (from vachaka), after the vachaka normalization.
  vachaka chose that clip with Su-śrotā, so the 0% of Su-śrotā is not a fair result.
- Moonshine gave no text for the 24-second clip A. It is good only for short segments.
- Su-śrotā writes only Devanagari, and it needs NeMo and torch on Python 3.12 or older. It is not an engine here.
  For Sanskrit speech, use Gemma or Whisper with the hint, or vachaka.

Gemma 4 E4B stays the default: it has no errors on clip B, it writes all 9 terms without a hint, and one 5-second segment takes 0.7 s.
For more speed, use Parakeet (0.15 s) or Qwen3-ASR 0.6B with the hint (0.14 s, all 9 terms).

`STT_SP` is the system prompt for the Gemma engine. It is the prompt `stt`, so you can change it on the Settings page. The other constants are the default models.
`ALIASES` keeps the old engine names: `whisper` and `parakeet` are the `mlx` engine with a fixed model.

In [ ]:
#| export
STT_SP = register('stt', 'Speech to text', 'How the Gemma speech engine writes down what you say. A change works when the model loads again.',
                  'You transcribe speech exactly. Keep technical terms, Sanskrit words (in IAST) and maths as spoken. Reply with the transcript only.')
DFLT_GEMMA = 'litert-community/gemma-4-E4B-it-litert-lm'
GEMMA_E2B = 'litert-community/gemma-4-E2B-it-litert-lm'
DFLT_WHISPER = 'mlx-community/whisper-large-v3-turbo-asr-fp16'
DFLT_PARAKEET = 'mlx-community/parakeet-tdt-0.6b-v3'
DFLT_OPENAI = 'gpt-4o-mini-transcribe'
DFLT_MODELS = dict(gemma=DFLT_GEMMA, mlx=DFLT_WHISPER, openai=DFLT_OPENAI)
ALIASES = dict(whisper=('mlx', DFLT_WHISPER), parakeet=('mlx', DFLT_PARAKEET))

## Settings

The settings are in `settings.json` in `cfg_dir()`. The `settings` module keeps the table of values and the file; this module uses its `stt` section.
The web app changes the settings, and the next segment uses the change. You do not have to restart.

In [ ]:
#| export
from abhyasa.settings import cfg_dir, DFLT_SETTINGS, settings, save_settings, update_settings, val

Without a file, you get the defaults. A change to one key keeps the other keys.

In [ ]:
test_eq({k: settings()['stt'][k] for k in ('engine', 'model', 'segment')}, dict(engine='gemma', model='', segment=True))
update_settings('stt', segment=False)
test_eq(val('stt.segment'), False)
update_settings('stt', segment=True)

`current` gives the engine and model in use. `$ABHYASA_STT` has priority over the settings. It can be an engine (`gemma`),
an old name (`whisper`), or an engine and a model (`mlx:mlx-community/Qwen3-ASR-0.6B-8bit`).

In [ ]:
#| export
def resolve(engine:str, model:str='') -> tuple:
    "The engine and model for a name: aliases become `mlx`, `engine:model` splits, an empty model gets the default."
    if ':' in engine: engine, model = engine.split(':', 1)
    if engine in ALIASES: engine, model = ALIASES[engine][0], model or ALIASES[engine][1]
    if engine not in DFLT_MODELS: raise ValueError(f'unknown speech engine {engine!r}')
    if engine == 'gemma': model = model or os.getenv('ABHYASA_GEMMA') or DFLT_GEMMA
    return engine, model or DFLT_MODELS[engine]

def current() -> tuple:
    "The engine and model in use: `$ABHYASA_STT`, else the settings."
    if (e := os.getenv('ABHYASA_STT')): return resolve(e)
    s = settings()['stt']
    return resolve(s['engine'], s['model'])

The settings give Gemma E4B by default. The environment variable has priority.

In [ ]:
test_eq(current(), ('gemma', DFLT_GEMMA))
test_eq(resolve('parakeet'), ('mlx', DFLT_PARAKEET))
test_eq(resolve('mlx:mlx-community/Qwen3-ASR-0.6B-8bit'), ('mlx', 'mlx-community/Qwen3-ASR-0.6B-8bit'))
os.environ['ABHYASA_STT'] = 'whisper'
test_eq(current(), ('mlx', DFLT_WHISPER))
del os.environ['ABHYASA_STT']

## One model in memory

`resident` keeps one speech model loaded. When the app asks for a different model, it frees the old model first.
Two large models in memory at the same time can make the Mac slow.

In [ ]:
#| export
_res, _lock = AttrDict(key=None, obj=None, close=None), threading.RLock()

def unload():
    "Free the loaded speech model."
    with _lock:
        if _res.close and _res.obj is not None: _res.close(_res.obj)
        _res.update(key=None, obj=None, close=None)
        gc.collect()
        try:
            import mlx.core as mx; mx.clear_cache()
        except ImportError: pass

def resident(key, load, close=None):
    "The loaded speech model for `key`: when `key` is new, free the old model, then `load()` the new one."
    with _lock:
        if _res.key != key:
            unload()
            _res.obj = load()
            _res.key, _res.close = key, close
        return _res.obj

A new key frees the old model. The same key does not load again.

In [ ]:
freed = []
resident('a', lambda: 'A', freed.append)
test_eq(resident('a', lambda: 'other'), 'A')
test_eq(resident('b', lambda: 'B'), 'B')
test_eq(freed, ['A'])
unload()

Each engine is a function that takes a WAV path, a hint and a model, and returns text.
Gemma keeps one LiteRT engine loaded, and makes a new conversation for each segment. Then one segment does not change the next.

In [ ]:
#| export
def _gemma_engine(model):
    from rishi.litert import LitertChat
    return resident(('gemma', model), lambda: LitertChat.create_engine(model).__enter__(), lambda e: e.__exit__(None, None, None))

def _gemma(path, hint='', model=DFLT_GEMMA):
    from rishi import Chat, resp_text
    chat = Chat(model, engine=_gemma_engine(model), sp=prompt('stt'))
    try: return resp_text(chat([f'Transcribe this clip.{" Words it may use: " + hint if hint else ""}', Path(path)]))
    finally: chat.close()

The `mlx` engine gives the hint to a model that accepts one: Whisper as `initial_prompt`, Qwen3-ASR as `hotwords`.

In [ ]:
#| export
def _hint_kw(m, hint:str) -> dict:
    "The `generate` argument that carries the hint, for models that accept one."
    ps = inspect.signature(m.generate).parameters
    if not hint: return {}
    if 'initial_prompt' in ps: return dict(initial_prompt=hint)
    if 'hotwords' in ps: return dict(hotwords=[o.strip() for o in hint.split(',') if o.strip()])
    return {}

def _mlx(path, hint='', model=DFLT_WHISPER):
    from mlx_audio.stt.utils import load
    m = resident(('mlx', model), lambda: load(model))
    return m.generate(str(path), **_hint_kw(m, hint)).text

A Whisper-like model gets `initial_prompt`. A Qwen3-like model gets a list of hot words. A model without either gets nothing.

In [ ]:
class _W:
    def generate(self, audio, initial_prompt=None): pass
class _Q:
    def generate(self, audio, hotwords=None): pass
test_eq(_hint_kw(_W(), 'Nyāya, hetu'), dict(initial_prompt='Nyāya, hetu'))
test_eq(_hint_kw(_Q(), 'Nyāya, hetu'), dict(hotwords=['Nyāya', 'hetu']))
test_eq(_hint_kw(_W(), ''), {})

## OpenAI

The `openai` engine posts the WAV to the OpenAI transcription API, with the hint as its `prompt`.
`http_post` is `httpx.post` unless a test puts a fake in its place.

In [ ]:
#| export
OPENAI_URL = 'https://api.openai.com/v1/audio/transcriptions'
http_post = None

def _openai(path, hint='', model=DFLT_OPENAI):
    key, _ = openai_key()
    if not key: raise RuntimeError('There is no OpenAI key. Set one on the Settings page.')
    post = http_post or __import__('httpx').post
    data = dict(model=model, response_format='json', **(dict(prompt=hint) if hint else {}))
    r = post(OPENAI_URL, headers={'Authorization': f'Bearer {key}'}, data=data,
             files=dict(file=('clip.wav', Path(path).read_bytes(), 'audio/wav')), timeout=60)
    r.raise_for_status()
    return r.json()['text']

ENGINES = dict(gemma=_gemma, mlx=_mlx, openai=_openai)

The key comes from the first of these places that has `OPENAI_API_KEY`: the environment, the abhyasa `keys.env`, then the ramabana `keys.env`. ramabana says where its file is (`$RAMABANA_CONFIG`, else `~/.config/ramabana/keys.env`).
abhyasa reads both files with ramabana's `load_keys`. It refuses a file that other users can read, and it runs `!command` values. abhyasa only reads the ramabana file. `set_openai_key` writes the abhyasa file with `save_key` (atomic, mode 0600).
ramabana is an optional extra. Without it, abhyasa reads and writes plain `NAME=value` lines.
`masked` shows only the last 4 characters.

In [ ]:
#| export
from fastcore.xtras import atomic_save
try: from ramabana.setup import keys_path, load_keys, save_key
except ImportError:   # ramabana is an optional extra: the same three names, for plain `NAME=value` lines
    def keys_path(): return Path('~/.config/ramabana/keys.env').expanduser()
    def load_keys(path=None):
        p, got = Path(path or keys_path()), []
        for l in (p.read_text().splitlines() if p.exists() and not p.stat().st_mode & 0o077 else []):
            k, _, v = l.strip().removeprefix('export ').partition('=')
            if k.strip() not in os.environ and v: os.environ[k.strip()] = v.strip().strip('"\''); got.append(k.strip())
        return got
    def save_key(name, value, path=None):
        p = Path(path or keys_path())
        keep = [l for l in (p.read_text().splitlines() if p.exists() else []) if l.strip().removeprefix('export ').partition('=')[0].strip() != name]
        with atomic_save(p, 'w') as f: f.write(''.join(l + '\n' for l in keep + [f'{name}={value}']))
        p.chmod(0o600); os.environ[name] = value; return p

def _file_key(p, name='OPENAI_API_KEY'):
    "`name` from the key file `p`, read by `load_keys`; no key stays in the environment."
    got = load_keys(p)
    k = os.environ.get(name, '') if name in got else ''
    for n in got: os.environ.pop(n, None)
    return k

def openai_key() -> tuple:
    "The OpenAI key and where it is from; `('', '')` when there is none."
    if (k := os.getenv('OPENAI_API_KEY')): return k, 'environment'
    for p in (cfg_dir()/'keys.env', keys_path()):
        if (k := _file_key(p)): return k, str(p)
    return '', ''

def set_openai_key(key:str=''):
    "Save `key` in abhyasa's `keys.env` (mode 0600); an empty key clears it. A key with other than letters, digits, `-` and `_` raises `ValueError`."
    if key.strip() and not re.fullmatch(r'[A-Za-z0-9_-]{8,300}', key.strip()): raise ValueError('An API key has only letters, digits, - and _.')
    old = os.environ.get('OPENAI_API_KEY')
    save_key('OPENAI_API_KEY', key.strip(), cfg_dir()/'keys.env')
    if old is None: os.environ.pop('OPENAI_API_KEY', None)   # `save_key` also sets the environment: only a file holds this key
    else: os.environ['OPENAI_API_KEY'] = old

def masked(key:str) -> str: return f'…{key[-4:]}' if key else ''

A key in the abhyasa file comes back with its source, and the file has mode 0600. When the abhyasa file has no key, abhyasa reads the ramabana file. This test points ramabana to a temp folder.

In [ ]:
os.environ['RAMABANA_CONFIG'] = str(Path(tempfile.mkdtemp())/'config.toml')
test_eq(openai_key(), ('', ''))
keys_path().write_text('export OPENAI_API_KEY="sk-ramabana-0000"\n'); keys_path().chmod(0o600)
test_eq(openai_key(), ('sk-ramabana-0000', str(keys_path())))
test_eq(os.getenv('OPENAI_API_KEY'), None)
set_openai_key('sk-test-1234')
test_eq(openai_key(), ('sk-test-1234', str(cfg_dir()/'keys.env')))
test_eq(oct((cfg_dir()/'keys.env').stat().st_mode & 0o777), '0o600')
test_eq(masked('sk-test-1234'), '…1234')

The engine sends the model, the hint and the WAV, with the key in the header. This test uses a fake HTTP client.

In [ ]:
class _R:
    def raise_for_status(self): pass
    def json(self): return dict(text='hetu and sādhya')
sent = {}
def fake_post(url, **kw): sent.update(url=url, **kw); return _R()
http_post = fake_post
p = Path(tempfile.mkdtemp())/'a.wav'; p.write_bytes(b'RIFF')
test_eq(_openai(p, 'hetu'), 'hetu and sādhya')
test_eq(sent['data'], dict(model=DFLT_OPENAI, response_format='json', prompt='hetu'))
test_eq(sent['headers']['Authorization'], 'Bearer sk-test-1234')
set_openai_key('')
keys_path().unlink()
test_fail(lambda: _openai(p), contains='no OpenAI key')
http_post = None

A key with a newline or a space can add a second line to the file. Thus `set_openai_key` refuses it.

In [ ]:
test_fail(lambda: set_openai_key('sk-a\nOTHER=1'), contains='only letters')
test_fail(lambda: set_openai_key('sk a b c d e'), contains='only letters')

## Transcribe

`transcribe` accepts a path or bytes. It writes bytes to a temporary file and removes the file after use.
Without an engine, it uses `current()`, so a change on the Settings page applies to the next segment.

In [ ]:
#| export
def transcribe(audio, engine:str=None, hint:str='', model:str=None) -> str:
    "Text for a WAV file or WAV bytes. `hint` lists words that the speaker can use, such as the name of the concept."
    engine, model = resolve(engine, model or '') if engine else current()
    eng = partial(ENGINES[engine], model=model)
    if isinstance(audio, (bytes, bytearray)):
        with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as f: f.write(audio)
        try: return eng(f.name, hint).strip()
        finally: os.unlink(f.name)
    return eng(audio, hint).strip()

`wav_seconds` gives the length of a clip in seconds. `silence` makes a clip of silence. The app uses this clip to warm a model.

In [ ]:
#| export
def wav_seconds(audio:bytes) -> float:
    "The length of WAV bytes, in seconds; 0 when they are not WAV."
    import io
    try:
        with wave.open(io.BytesIO(audio)) as w: return w.getnframes() / w.getframerate()
    except Exception: return 0.

def silence(secs:float=.5) -> bytes:
    "WAV bytes of silence: 16 kHz, mono, 16 bit."
    import io
    b = io.BytesIO()
    with wave.open(b, 'wb') as w: w.setnchannels(1); w.setsampwidth(2); w.setframerate(16000); w.writeframes(b'\0\0' * int(16000 * secs))
    return b.getvalue()

Half a second of silence is 0.5 seconds. Bytes that are not WAV give 0.

In [ ]:
test_eq(wav_seconds(silence()), .5)
test_eq(wav_seconds(b'nope'), 0.)

## Models

`KNOWN` lists the models that we measured (see the table above). The Settings page shows them first.
`cached` tells if a model is in the Hugging Face cache. It does not use the network.

In [ ]:
#| export
KNOWN = [
    dict(engine='gemma', model=DFLT_GEMMA, bench={'seg5': 0.69, 'wer_a': 0.086, 'wer_b': 0.0, 'terms': 9, 'mem': 1.5}, label='Gemma 4 E4B (LiteRT)', note='the default; good with IAST in the hint'),
    dict(engine='gemma', model=GEMMA_E2B, bench={'seg5': 0.38, 'wer_a': 0.143, 'wer_b': 0.021, 'terms': 8, 'mem': 1.3}, label='Gemma 4 E2B (LiteRT)', note='faster than E4B'),
    dict(engine='mlx', model=DFLT_WHISPER, bench={'seg5': 0.68, 'wer_a': 0.057, 'wer_b': 0.0, 'terms': 9, 'mem': 2.3}, label='Whisper large-v3-turbo', note='reads the hint as its initial prompt'),
    dict(engine='mlx', model=DFLT_PARAKEET, bench={'seg5': 0.15, 'wer_a': 0.057, 'wer_b': 0.043, 'terms': 7, 'mem': 3.2}, label='Parakeet TDT 0.6B v3', note='the fastest; European languages only'),
    dict(engine='mlx', model='mlx-community/Qwen3-ASR-0.6B-8bit', bench={'seg5': 0.14, 'wer_a': 0.057, 'wer_b': 0.0, 'terms': 9, 'mem': 2.0}, label='Qwen3-ASR 0.6B (8 bit)', note='reads the hint as hot words'),
    dict(engine='mlx', model='mlx-community/Qwen3-ASR-1.7B-8bit', bench={'seg5': 0.25, 'wer_a': 0.057, 'wer_b': 0.0, 'terms': 9, 'mem': 3.6}, label='Qwen3-ASR 1.7B (8 bit)', note='reads the hint as hot words'),
    dict(engine='mlx', model='mlx-community/Voxtral-Mini-4B-Realtime-2602-4bit', bench={'seg5': 2.21, 'wer_a': 0.057, 'wer_b': 0.064, 'terms': 7, 'mem': 5.5}, label='Voxtral Mini 4B Realtime (4 bit)', note='made for live speech'),
    dict(engine='mlx', model='UsefulSensors/moonshine-base', bench={'seg5': 0.07, 'wer_a': 1.0, 'wer_b': 0.085, 'terms': 6, 'mem': 0.8}, label='Moonshine base', note='small; English only; short segments only'),
    dict(engine='mlx', model='Mediform/canary-1b-v2-mlx-q8', bench={'seg5': 0.08, 'wer_a': 0.057, 'wer_b': 0.043, 'terms': 8, 'mem': 1.4}, label='Canary 1B v2 (8 bit)', note='European languages'),
    dict(engine='openai', model=DFLT_OPENAI, label='OpenAI gpt-4o-mini-transcribe', note='sends audio off this machine; needs a key'),
    dict(engine='openai', model='gpt-4o-transcribe', label='OpenAI gpt-4o-transcribe', note='sends audio off this machine; needs a key'),
]

def cached(model:str) -> bool:
    "True when `model` is in the Hugging Face cache."
    from huggingface_hub.constants import HF_HUB_CACHE
    d = Path(HF_HUB_CACHE)/f"models--{model.replace('/', '--')}"/'snapshots'
    return d.is_dir() and any(d.iterdir())

def cache_bytes(model:str) -> int:
    "The bytes of `model` in the Hugging Face cache so far, with the files that are still downloading."
    from huggingface_hub.constants import HF_HUB_CACHE
    d = Path(HF_HUB_CACHE)/f"models--{model.replace('/', '--')}"/'blobs'
    return sum(f.stat().st_size for f in d.iterdir()) if d.is_dir() else 0

`search_models` looks up mlx-community speech to text models on Hugging Face, with their size. It drops text to speech, aligners
and diarization models, which the `mlx` engine cannot use.

In [ ]:
#| export
STT_WORDS = ('whisper', 'parakeet', 'asr', 'voxtral', 'canary', 'moonshine', 'granite', 'sensevoice', 'nemotron', 'stt', 'speech')
NOT_STT = ('tts', 'aligner', 'diariz', 'music', 'vad', 'speech-tokenizer')

def is_stt(model:str) -> bool:
    "True when the repo name looks like a speech to text model."
    m = model.lower()
    return any(w in m for w in STT_WORDS) and not any(w in m for w in NOT_STT)

def model_size(model:str, api=None) -> int:
    "The bytes of all the files of `model` on Hugging Face."
    from huggingface_hub import HfApi
    return sum(s.size or 0 for s in (api or HfApi()).model_info(model, files_metadata=True).siblings)

def search_models(q:str='', limit:int=15, api=None) -> list:
    "mlx-community speech to text models that match `q`, most downloaded first, with their size."
    from huggingface_hub import HfApi
    api = api or HfApi()
    ms = [m for m in api.list_models(author='mlx-community', search=q or None, sort='downloads', limit=400) if is_stt(m.id)][:limit]
    with ThreadPoolExecutor(8) as ex: sizes = list(ex.map(lambda m: model_size(m.id, api), ms))
    return [dict(model=m.id, downloads=m.downloads or 0, size=s) for m, s in zip(ms, sizes)]

The filter keeps speech to text and drops the others.

In [ ]:
test_eq([is_stt(m) for m in ('mlx-community/Qwen3-ASR-0.6B-8bit', 'mlx-community/whisper-tiny-asr-fp16', 'mlx-community/Qwen3-TTS-12Hz-1.7B-Base-bf16', 'mlx-community/silero-vad')],
        [True, True, False, False])

`fetch` downloads a model and notes it in the settings. `remove` deletes a model from the cache, but only a model that `fetch` downloaded.
Other entries in the cache belong to other programs.

In [ ]:
#| export
def fetch(model:str):
    "Download `model` into the Hugging Face cache; note it when it was not there before."
    from huggingface_hub import snapshot_download
    new = not cached(model)
    snapshot_download(model)
    if new:
        d = settings(); d['downloaded'] = sorted({*d['downloaded'], model}); save_settings(d)

def remove(model:str):
    "Delete `model` from the Hugging Face cache. Only a model that `fetch` downloaded."
    d = settings()
    if model not in d['downloaded']: raise ValueError(f'{model} was not downloaded by abhyasa, so it stays')
    if _res.key and _res.key[1] == model: unload()
    from huggingface_hub import scan_cache_dir
    info = scan_cache_dir()
    revs = [r.commit_hash for repo in info.repos if repo.repo_id == model for r in repo.revisions]
    if revs: info.delete_revisions(*revs).execute()
    d['downloaded'].remove(model); save_settings(d)

`remove` refuses a model that abhyasa did not download.

In [ ]:
test_fail(lambda: remove(DFLT_GEMMA), contains='stays')

## Segments in order

The browser sends one segment each time you stop to think. `Speech` has one worker thread for all the model calls.
Thus only one model is in memory, and the segments run one at a time, in the order they came.
A `Dictation` collects the segments of one answer. `next` gives the text in the order of the segment numbers,
also when a slow request arrives late. When the browser sends a segment again (after an error), the code does not use it two times.

In [ ]:
#| export
class Dictation:
    "The segments of one spoken answer: each is transcribed on the speech worker, and the text comes out in order."
    def __init__(self, speech, hint=''):
        store_attr()
        self.seen, self.ready, self.out, self.n_done, self.total, self.t = set(), {}, 0, 0, None, time.time()
        self.cv = threading.Condition()

    def add(self, seq:int, audio:bytes) -> bool:
        "Queue segment `seq`; False when it came before."
        with self.cv:
            if seq in self.seen: return False
            self.seen.add(seq)
        self.speech.submit(self._run, seq, audio)
        return True

    def _run(self, seq, audio):
        try: r = dict(seq=seq, text=self.speech.fn(audio, hint=self.hint).strip())
        except Exception as e: r = dict(seq=seq, text='', error=str(e))
        with self.cv: self.ready[seq] = r; self.n_done += 1; self.cv.notify_all()

    def stop(self, n:int):
        "There are `n` segments in all."
        with self.cv: self.total = n; self.cv.notify_all()

    @property
    def pending(self) -> int: return len(self.seen) - self.n_done

    def next(self, timeout=None):
        "The next result in order; `dict(done=True)` after the last one; None at the timeout."
        with self.cv:
            if not self.cv.wait_for(lambda: self.out in self.ready or self.total == self.out, timeout): return None
            if self.out not in self.ready: return dict(done=True)
            self.out += 1
            return self.ready.pop(self.out - 1)

`Speech` owns the worker. `warm` loads the model in use with half a second of silence, so the first segment does not wait for it.
`use` changes the engine and model, saves the change, and warms the new model.

In [ ]:
#| export
class Speech:
    "One worker thread for every speech model call, and the open dictations."
    def __init__(self, fn=None):
        self.fn, self.runs, self.status = fn or transcribe, {}, 'not loaded'
        self.ex = ThreadPoolExecutor(1, thread_name_prefix='speech')

    def submit(self, f, *a, **kw): return self.ex.submit(f, *a, **kw)

    def warm(self):
        "Load the model in use on the worker."
        def go():
            try: self.fn(silence(), hint=''); self.status = 'ready'
            except Exception as ex: self.status = f'failed: {ex}'
        self.status = 'loading'   # at once, so a page drawn now shows it
        return self.submit(go)

    def use(self, engine:str, model:str=''):
        "Switch to `engine` and `model`, save the choice, and warm the new model."
        engine, model = resolve(engine, model)
        update_settings('stt', engine=engine, model=model)
        return self.warm()

    def start(self, hint:str='') -> str:
        "Open a dictation; returns its id. Dictations older than an hour are dropped."
        for k in [k for k, d in self.runs.items() if time.time() - d.t > 3600]: self.runs.pop(k, None)
        did = uuid.uuid4().hex[:10]
        self.runs[did] = Dictation(self, hint)
        return did

### Tests with a fake engine

This fake engine is slow on even segments, so the requests finish out of order. The text must still come out in order.

In [ ]:
calls = []
def slow_fake(audio, hint=''):
    n = int(audio.decode())
    calls.append(n)
    time.sleep(.05 if n % 2 == 0 else 0)
    return f'segment {n} ({hint})'
sp = Speech(slow_fake)
d = sp.runs[sp.start('hetu')]

Segments 0 to 5 arrive out of order, from several threads. Segment 3 comes two times.

In [ ]:
order = [2, 0, 1, 3, 3, 5, 4]
with ThreadPoolExecutor(4) as ex: added = list(ex.map(lambda n: d.add(n, str(n).encode()), order))
test_eq(sum(added), 6)
d.stop(6)
out = []
while not (r := d.next(5)).get('done'): out.append(r)
test_eq([r['seq'] for r in out], list(range(6)))
test_eq(out[0]['text'], 'segment 0 (hetu)')
test_eq(sorted(calls), list(range(6)))

All the model calls ran on the one worker thread, one at a time.

In [ ]:
seen = set()
sp.submit(lambda: seen.add(threading.current_thread().name)).result()
test_eq(len(seen), 1)
test_eq(d.pending, 0)

An engine error gives an empty text with the error. It does not stop the segments after it.

In [ ]:
def flaky(audio, hint=''):
    if audio == b'bad': raise RuntimeError('no model')
    return 'ok'
sp = Speech(flaky)
d = sp.runs[sp.start()]
d.add(0, b'bad'); d.add(1, b'good'); d.stop(2)
test_eq(d.next(5), dict(seq=0, text='', error='no model'))
test_eq(d.next(5), dict(seq=1, text='ok'))
test_eq(d.next(5), dict(done=True))

Without `stop`, `next` waits and then gives None. `use` saves the choice and warms the new model with silence.

In [ ]:
d = sp.runs[sp.start()]
test_is(d.next(.05), None)
heard = []
sp = Speech(lambda a, hint='': heard.append(wav_seconds(a)) or '')
sp.use('parakeet').result()
test_eq(current(), ('mlx', DFLT_PARAKEET))
test_eq(heard, [.5])
assert sp.status.startswith('ready'), sp.status
update_settings('stt', engine='gemma', model='')

## Real models

These tests use a real clip and real models. They are slow. Run them with `nbdev-test --flags slow`.

In [ ]:
#| slow
clip = first(p for p in (Path('../../rishi/nbs/speech.wav'), Path('../../../rishi/nbs/speech.wav')) if p.exists())   # from the repo or a worktree
if clip:
    for e in ('gemma', 'parakeet', 'mlx:mlx-community/Qwen3-ASR-0.6B-8bit'): print(e, transcribe(clip, e, hint='Tennyson'))
    unload()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()